# External Agents CRUD — `@azure/ai-projects`

Demonstrates CRUD operations on external agents using the `AIProjectClient`. External Agents are a preview feature that register third-party agents hosted outside Microsoft Foundry; registration is metadata-only, using the OpenTelemetry agent identifier to light up traces and evaluations for spans emitted by your external agent.

It mirrors the [`externalAgentsCrud.ts`](./externalAgentsCrud.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. Build the package: run `pnpm build` in `sdk/ai/ai-projects/`.
2. Install and register the [tslab](https://github.com/yunabe/tslab) kernel (`npm install -g tslab && tslab install`), then select the **TypeScript** kernel for this notebook.
3. Launch VS Code / Jupyter from `sdk/ai/ai-projects/`.
4. Sign in with `az login`.
5. **Environment variables** (read by this notebook via `process.env`):
   - `FOUNDRY_PROJECT_ENDPOINT` — your Foundry project endpoint.

Run the cells in order (top to bottom); state is shared across cells.

In [10]:
import type { ExternalAgentDefinition } from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

require("dotenv").config({ path: require("path").resolve(process.cwd(), "../../.env") });

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";


In [3]:
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

const agentName = "sample-external-agent";
const otelAgentId = "sample-external-agent";

In [4]:
// Clean up any leftover agent from a previous run.
try {
  await project.agents.delete(agentName, { force: true });
  console.log(`External agent \`${agentName}\` deleted`);
} catch (error: any) {
  if (error.statusCode !== 404) {
    throw error;
  }
}

In [5]:
// Create an external agent version. External agents are a preview feature,
// so the `ExternalAgents=V1Preview` opt-in is required.
const created = await project.agents.createVersion(
  agentName,
  {
    kind: "external",
    otel_agent_id: otelAgentId,
  } as ExternalAgentDefinition,
  {
    foundryFeatures: "ExternalAgents=V1Preview",
    description: "External agent registered by the @azure/ai-projects sample.",
    metadata: { sample: "external_agents_crud", status: "created" },
  },
);
console.log(
  `Created external agent: ${created.name} version=${created.version} otel_agent_id=${otelAgentId}`,
);

Created external agent: sample-external-agent version=1 otel_agent_id=sample-external-agent


In [6]:
// Retrieve the agent by name (latest version).
const fetchedAgent = await project.agents.get(agentName);
console.log(
  `Retrieved external agent: ${fetchedAgent.name} latest_version=${fetchedAgent.versions.latest.version}`,
);

Retrieved external agent: sample-external-agent latest_version=1


In [7]:
// Retrieve a specific version of the agent.
const fetchedVersion = await project.agents.getVersion(agentName, created.version);
console.log(
  `Retrieved external agent version: ${fetchedVersion.name} version=${fetchedVersion.version}`,
);

Retrieved external agent version: sample-external-agent version=1


In [8]:
// List external agents.
const externalAgents = [];
const iterator = project.agents.list({ kind: "external", limit: 10 })[Symbol.asyncIterator]();
let next = await iterator.next();
while (!next.done) {
  const externalAgent = next.value;
  externalAgents.push(externalAgent);
  next = await iterator.next();
}
console.log(`Found ${externalAgents.length} external agents or more`);
for (const externalAgent of externalAgents) {
  console.log(`  - ${externalAgent.name} (${externalAgent.id})`);
}

Found 1 external agents or more
  - sample-external-agent (sample-external-agent)
  - sample-external-agent (sample-external-agent)


In [9]:
// Delete the external agent.
const deleted = await project.agents.delete(agentName, { force: true });
console.log(`Deleted external agent: ${deleted.name} deleted=${deleted.deleted}`);

Deleted external agent: sample-external-agent deleted=true
